In [2]:
!pip install albumentations -q
print("Done!")

Done!


In [3]:
import os

os.makedirs('/kaggle/working/src',     exist_ok=True)
os.makedirs('/kaggle/working/models',  exist_ok=True)
os.makedirs('/kaggle/working/outputs', exist_ok=True)

print("Folders created!")

Folders created!


In [4]:
%%writefile /kaggle/working/src/model.py

import torch
import torch.nn as nn


class ConvBlock(nn.Module):
    # two consecutive conv layers each followed by batch norm and relu
    def __init__(self, in_channels, out_channels):
        super(ConvBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels,
                      kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.block(x)


class EncoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(EncoderBlock, self).__init__()
        self.conv = ConvBlock(in_channels, out_channels)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        skip   = self.conv(x)
        pooled = self.pool(skip)
        return pooled, skip


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(DecoderBlock, self).__init__()
        self.upsample = nn.ConvTranspose2d(in_channels, out_channels,
                                            kernel_size=2, stride=2)
        self.conv = ConvBlock(out_channels * 2, out_channels)

    def forward(self, x, skip):
        x = self.upsample(x)
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


class UNet(nn.Module):
    """
    U-Net for binary brain tumour segmentation.
    Input:  (batch, 3, 256, 256)
    Output: (batch, 1, 256, 256)
    """
    def __init__(self, in_channels=3, out_channels=1,
                 features=[64, 128, 256, 512]):
        super(UNet, self).__init__()

        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()

        ch = in_channels
        for f in features:
            self.encoders.append(EncoderBlock(ch, f))
            ch = f

        self.bottleneck = ConvBlock(features[-1], features[-1] * 2)

        for f in reversed(features):
            self.decoders.append(DecoderBlock(f * 2, f))

        self.final_conv = nn.Conv2d(features[0], out_channels,
                                     kernel_size=1)

    def forward(self, x):
        skips = []

        for encoder in self.encoders:
            x, skip = encoder(x)
            skips.append(skip)

        x = self.bottleneck(x)

        skips = skips[::-1]
        for i, decoder in enumerate(self.decoders):
            x = decoder(x, skips[i])

        return self.final_conv(x)

Writing /kaggle/working/src/model.py


In [6]:
%%writefile /kaggle/working/src/dataset.py

import os
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset
import albumentations as A
from albumentations.pytorch import ToTensorV2


class BrainMRIDataset(Dataset):
    def __init__(self, image_paths, mask_paths,
                 img_size=256, augment=False):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.img_size    = img_size
        self.augment     = augment
        self.transform   = self._build_transforms()

    def _build_transforms(self):
        if self.augment:
            return A.Compose([
                A.Resize(self.img_size, self.img_size),
                A.HorizontalFlip(p=0.5),
                A.VerticalFlip(p=0.2),
                A.Rotate(limit=15, p=0.4),
                A.ElasticTransform(
                    alpha=120, sigma=120 * 0.05, p=0.3
                ),
                A.RandomBrightnessContrast(
                    brightness_limit=0.2,
                    contrast_limit=0.2, p=0.4
                ),
                A.GaussianBlur(blur_limit=3, p=0.2),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                ToTensorV2()
            ])
        else:
            return A.Compose([
                A.Resize(self.img_size, self.img_size),
                A.Normalize(
                    mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225]
                ),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img  = cv2.imread(self.image_paths[idx])
        img  = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(self.mask_paths[idx],
                          cv2.IMREAD_GRAYSCALE)
        mask = (mask > 0).astype(np.float32)

        transformed = self.transform(image=img, mask=mask)
        img  = transformed['image']
        mask = transformed['mask']
        mask = mask.unsqueeze(0)

        return img, mask

Overwriting /kaggle/working/src/dataset.py


In [7]:
%%writefile /kaggle/working/src/evaluate.py

import torch


def dice_coefficient(pred, target, threshold=0.5, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    pred   = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    dice = (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return dice.item()


def iou_score(pred, target, threshold=0.5, smooth=1e-6):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()

    pred   = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    union        = pred.sum() + target.sum() - intersection

    iou = (intersection + smooth) / (union + smooth)

    return iou.item()


def dice_loss(pred, target, smooth=1e-6):
    pred = torch.sigmoid(pred)

    pred   = pred.view(-1)
    target = target.view(-1)

    intersection = (pred * target).sum()
    dice = (2.0 * intersection + smooth) / (pred.sum() + target.sum() + smooth)

    return 1.0 - dice

Writing /kaggle/working/src/evaluate.py


In [8]:
%%writefile /kaggle/working/src/train.py

import os
import torch
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split

from src.dataset  import BrainMRIDataset
from src.model    import UNet
from src.evaluate import dice_coefficient, iou_score, dice_loss


def train_one_epoch(model, loader, optimiser, device):
    model.train()
    total_loss = 0.0
    total_dice = 0.0

    for imgs, masks in loader:
        imgs  = imgs.to(device)
        masks = masks.to(device)

        optimiser.zero_grad()
        preds = model(imgs)
        loss  = dice_loss(preds, masks)
        loss.backward()
        optimiser.step()

        total_loss += loss.item()
        total_dice += dice_coefficient(preds.detach(), masks)

    return total_loss / len(loader), total_dice / len(loader)


def validate(model, loader, device):
    model.eval()
    total_loss = 0.0
    total_dice = 0.0
    total_iou  = 0.0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs  = imgs.to(device)
            masks = masks.to(device)
            preds = model(imgs)

            total_loss += dice_loss(preds, masks).item()
            total_dice += dice_coefficient(preds, masks)
            total_iou  += iou_score(preds, masks)

    n = len(loader)
    return total_loss/n, total_dice/n, total_iou/n


def train(image_paths, mask_paths,
          epochs=30, batch_size=16, lr=1e-4,
          img_size=256,
          save_path='/kaggle/working/models/best_model.pth',
          device=None):

    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available()
                              else 'cpu')

    print(f"Training on: {device}")
    print(f"Epochs:      {epochs}")
    print(f"Batch size:  {batch_size}")
    print(f"LR:          {lr}")
    print()

    train_imgs, val_imgs, train_masks, val_masks = train_test_split(
        image_paths, mask_paths,
        test_size=0.2, random_state=42
    )

    print(f"Train samples: {len(train_imgs)}")
    print(f"Val samples:   {len(val_imgs)}")

    train_dataset = BrainMRIDataset(train_imgs, train_masks,
                                     img_size=img_size, augment=True)
    val_dataset   = BrainMRIDataset(val_imgs,   val_masks,
                                     img_size=img_size, augment=False)

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                              shuffle=True,  num_workers=2)
    val_loader   = DataLoader(val_dataset,   batch_size=batch_size,
                              shuffle=False, num_workers=2)

    model     = UNet(in_channels=3, out_channels=1).to(device)
    optimiser = torch.optim.Adam(model.parameters(), lr=lr)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimiser, mode='min', patience=5, factor=0.5
    )

    best_dice = 0.0
    history   = []

    os.makedirs(os.path.dirname(save_path), exist_ok=True)

    for epoch in range(1, epochs + 1):
        train_loss, train_dice = train_one_epoch(
            model, train_loader, optimiser, device
        )
        val_loss, val_dice, val_iou = validate(
            model, val_loader, device
        )

        scheduler.step(val_loss)

        history.append({
            'epoch'     : epoch,
            'train_loss': round(train_loss, 4),
            'train_dice': round(train_dice, 4),
            'val_loss'  : round(val_loss,   4),
            'val_dice'  : round(val_dice,   4),
            'val_iou'   : round(val_iou,    4),
        })

        print(f"Epoch {epoch:3d}/{epochs} | "
              f"train_loss: {train_loss:.4f} | "
              f"train_dice: {train_dice:.4f} | "
              f"val_loss: {val_loss:.4f} | "
              f"val_dice: {val_dice:.4f} | "
              f"val_iou: {val_iou:.4f}")

        if val_dice > best_dice:
            best_dice = val_dice
            torch.save({
                'epoch'      : epoch,
                'model_state': model.state_dict(),
                'val_dice'   : val_dice,
                'val_iou'    : val_iou,
            }, save_path)
            print(f"  → best model saved (val_dice={val_dice:.4f})")

    print(f"\nTraining complete. Best val_dice: {best_dice:.4f}")
    return model, history

Writing /kaggle/working/src/train.py


In [10]:
import sys
sys.path.insert(0, '/kaggle/working')

import os
import cv2
import torch

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

# collect all image mask pairs
data_path = '/kaggle/input/datasets/mateuszbuda/lgg-mri-segmentation/kaggle_3m'
patients  = os.listdir(data_path)

images = []
masks  = []

for patient in patients:
    patient_path = os.path.join(data_path, patient)
    if not os.path.isdir(patient_path):
        continue
    for fname in os.listdir(patient_path):
        if fname.endswith('.tif') and 'mask' not in fname:
            img_path  = os.path.join(patient_path, fname)
            mask_path = img_path.replace('.tif', '_mask.tif')
            if os.path.exists(mask_path):
                images.append(img_path)
                masks.append(mask_path)

print(f"Total pairs: {len(images)}")

# run full training
from src.train import train

model, history = train(
    image_paths = images,
    mask_paths  = masks,
    epochs      = 50,
    batch_size  = 16,
    lr          = 1e-4,
    img_size    = 256,
    save_path   = '/kaggle/working/models/best_model.pth'
)

GPU: Tesla T4
VRAM: 15.6 GB
Total pairs: 3929
Training on: cuda
Epochs:      50
Batch size:  16
LR:          0.0001

Train samples: 3143
Val samples:   786
Epoch   1/50 | train_loss: 0.9497 | train_dice: 0.1594 | val_loss: 0.9295 | val_dice: 0.2872 | val_iou: 0.1744
  → best model saved (val_dice=0.2872)
Epoch   2/50 | train_loss: 0.9299 | train_dice: 0.2644 | val_loss: 0.9027 | val_dice: 0.4261 | val_iou: 0.2778
  → best model saved (val_dice=0.4261)
Epoch   3/50 | train_loss: 0.8993 | train_dice: 0.3466 | val_loss: 0.8811 | val_dice: 0.2475 | val_iou: 0.1437
Epoch   4/50 | train_loss: 0.8442 | train_dice: 0.4484 | val_loss: 0.7876 | val_dice: 0.4899 | val_iou: 0.3378
  → best model saved (val_dice=0.4899)
Epoch   5/50 | train_loss: 0.7517 | train_dice: 0.5378 | val_loss: 0.7045 | val_dice: 0.5093 | val_iou: 0.3556
  → best model saved (val_dice=0.5093)
Epoch   6/50 | train_loss: 0.6258 | train_dice: 0.6218 | val_loss: 0.5400 | val_dice: 0.6778 | val_iou: 0.5299
  → best model saved (